# Notebook Overview — Combine and Split

## Purpose

This notebook combines six preprocessed dataset metadata CSV files and creates balanced **train** and **test** split CSV files for downstream model training and evaluation.

---

## Inputs

The notebook expects six preprocessed metadata CSV files generated by the previous stage:

* `metadata/preprocessed/imgn_preprocessed_metadata.csv`
* `metadata/preprocessed/coco_preprocessed_metadata.csv`
* `metadata/preprocessed/open_preprocessed_metadata.csv`
* `metadata/preprocessed/diff_preprocessed_metadata.csv`
* `metadata/preprocessed/sdxl_preprocessed_metadata.csv`
* `metadata/preprocessed/midj_preprocessed_metadata.csv`

All paths and filenames are defined in `project_config.py`.

---

## Assumptions

* Each input CSV contains a `filename` column
* Each input CSV contains **3000 rows**
* Dataset class assignments:

  | Dataset            | Class |
  |--------------------|-------|
  | ImageNet_1K_256    | rl    |
  | MS_COCO_2017       | rl    |
  | OpenImages         | rl    |
  | DiffusionDB        | ai    |
  | SDXL_Generated_10K | ai    |
  | Midjourney         | ai    |

* This notebook does **not**:
  - move images  
  - copy data  
  - unzip archives  
  - preprocess images  

* All operations are performed on metadata only

---

## Outputs

The following files are written to the project metadata directory:

* `metadata/splits/combined_metadata.csv`
* `metadata/splits/train_metadata.csv`
* `metadata/splits/test_metadata.csv`

---

## Expected Sizes

* `combined_metadata.csv` → **18,000 rows**
* `train_metadata.csv` → **14,400 rows**
* `test_metadata.csv` → **3,600 rows**

---

## Notes

* The split is **balanced by dataset and class label**
* A fixed random seed ensures **reproducibility**
* The training dataset is intended for **k-fold cross-validation** in later stages
* Image files remain in their original storage locations; metadata defines dataset structure
* This notebook represents the first stage that operates entirely on structured metadata

---

**Next step:** Proceed to feature extraction using the generated train/test metadata.
---
---

### Step 1 — Environment Setup

* Imports project configuration from `project_config.py`
* Defines runtime paths
* Verifies that all required metadata CSV files exist

In [ ]:
# ============================================
# Step 1: Environment Setup and Configuration
# ============================================

import os
import sys
import importlib.util
from pathlib import Path

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # User toggle (True or False)

# -------------------------------------------------
# Define project locations
# -------------------------------------------------
BASE_DIR = "/content/dip-ai-image-detection"
PROJECT_SRC_DIR = f"{BASE_DIR}/src"
CONFIG_FILE = f"{PROJECT_SRC_DIR}/project_config.py"

print("Initializing environment...")

# -------------------------------------------------
# Clone repo if not already present
# -------------------------------------------------
if not os.path.isdir(BASE_DIR):
    print("Cloning repository...")
    !git clone -q https://github.com/pgailinas/dip-ai-image-detection.git
else:
    print("Repository already exists. Skipping clone.")

# -------------------------------------------------
# Verify structure
# -------------------------------------------------
if not os.path.isdir(PROJECT_SRC_DIR):
    raise FileNotFoundError(f"Missing directory: {PROJECT_SRC_DIR}")

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(f"Missing file: {CONFIG_FILE}")

# -------------------------------------------------
# Add src to path
# -------------------------------------------------
if PROJECT_SRC_DIR not in sys.path:
    sys.path.insert(0, PROJECT_SRC_DIR)

# -------------------------------------------------
# Import config
# -------------------------------------------------
spec = importlib.util.spec_from_file_location("project_config", CONFIG_FILE)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

# -------------------------------------------------
# Pull only what this notebook actually uses
# -------------------------------------------------
AI_LABEL = cfg.AI_LABEL
REAL_LABEL = cfg.REAL_LABEL

SOURCE_LABEL_MAP = cfg.SOURCE_LABEL_MAP
VALID_SOURCES = cfg.VALID_SOURCES
REAL_SOURCES = cfg.REAL_SOURCES
AI_SOURCES = cfg.AI_SOURCES

SOURCE_FOLDER_NAMES = cfg.SOURCE_FOLDER_NAMES
PREPROCESSED_METADATA_FILES = cfg.PREPROCESSED_METADATA_FILES

PREPROCESSED_METADATA_DIR = cfg.PREPROCESSED_METADATA_DIR
SPLITS_METADATA_DIR = cfg.SPLITS_METADATA_DIR

TRAIN_SUBSET = cfg.TRAIN_SUBSET
TEST_SUBSET = cfg.TEST_SUBSET

TRAIN_PER_SOURCE = cfg.TRAIN_PER_SOURCE
TEST_PER_SOURCE = cfg.TEST_PER_SOURCE
RANDOM_SEED = cfg.RANDOM_SEED

COMBINED_METADATA_FILENAME = cfg.COMBINED_METADATA_FILENAME
TRAIN_METADATA_FILENAME = cfg.TRAIN_METADATA_FILENAME
TEST_METADATA_FILENAME = cfg.TEST_METADATA_FILENAME

COMBINED_METADATA_PATH = cfg.COMBINED_METADATA_PATH
TRAIN_METADATA_PATH = cfg.TRAIN_METADATA_PATH
TEST_METADATA_PATH = cfg.TEST_METADATA_PATH

# -------------------------------------------------
# Define notebook paths
# -------------------------------------------------
INPUT_CSV_DIR = Path(PREPROCESSED_METADATA_DIR)
OUTPUT_DIR = Path(SPLITS_METADATA_DIR)

# -------------------------------------------------
# Ensure required output directory exists
# -------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
SOURCE_ORDER = cfg.REAL_SOURCES + cfg.AI_SOURCES

INPUT_FILES = [
    Path(PREPROCESSED_METADATA_FILES[source])
    for source in SOURCE_ORDER
]

print("Verifying required metadata CSV files...\n")

missing_files = []

for csv_path in INPUT_FILES:
    if not csv_path.exists():
        missing_files.append(str(csv_path))

if missing_files:
    raise FileNotFoundError(
        f"Missing required preprocessed metadata files: {missing_files}"
    )

print("All required metadata CSV files are present.")
print("\nEnvironment setup complete.\n")

if VERBOSE:
    print(f"BASE_DIR                  : {cfg.BASE_DIR}")
    print(f"PROJECT_SRC_DIR           : {PROJECT_SRC_DIR}")
    print(f"PREPROCESSED_METADATA_DIR : {PREPROCESSED_METADATA_DIR}")
    print(f"SPLITS_METADATA_DIR       : {SPLITS_METADATA_DIR}")
    print(f"INPUT_CSV_DIR             : {INPUT_CSV_DIR}")
    print(f"OUTPUT_DIR                : {OUTPUT_DIR}")
    print(f"TRAIN_PER_SOURCE          : {TRAIN_PER_SOURCE}")
    print(f"TEST_PER_SOURCE           : {TEST_PER_SOURCE}")
    print(f"RANDOM_SEED               : {RANDOM_SEED}")


### Step 2 — Define Paths

* Defines input and output directories using config-based paths

In [ ]:
# =========================
# Step 2: Define Paths
# =========================

import pandas as pd

if VERBOSE:
    print("Paths defined.")
    print(f"INPUT_CSV_DIR = {INPUT_CSV_DIR}")
    print(f"OUTPUT_DIR    = {OUTPUT_DIR}")



### Step 3 — Dataset Configuration

* Builds dataset configuration dynamically from `project_config.py`
* Associates each dataset with:
  - metadata CSV file
  - class label

In [ ]:
# =========================
# Step 3: Dataset configuration
# =========================

DATASET_CONFIG = {}

for source in VALID_SOURCES:
    dataset_name = SOURCE_FOLDER_NAMES[source]

    DATASET_CONFIG[dataset_name] = {
        "csv": Path(PREPROCESSED_METADATA_FILES[source]).name,
        "class_label": SOURCE_LABEL_MAP[source],
        "source_key": source,   # useful later
    }

print("Dataset configurations defined.")

if VERBOSE:
    for name, cfg_entry in DATASET_CONFIG.items():
        print(f"{name}:")
        print(f"  csv         : {cfg_entry['csv']}")
        print(f"  class_label : {cfg_entry['class_label']}")


### Step 4 — Combine Metadata

* Loads all six metadata CSV files
* Extracts required columns
* Adds:
  - `source_dataset`
  - `class_label`
* Combines into a single DataFrame
* Saves:
  - `combined_metadata.csv`

In [ ]:
# =========================
# Step 4: Load and combine metadata
# =========================

dfs = []

BASE_COLUMNS = ["filename"]

for dataset_name, dataset_cfg in DATASET_CONFIG.items():
    csv_path = INPUT_CSV_DIR / dataset_cfg["csv"]

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    if "filename" not in df.columns:
        raise ValueError(f"'filename' column not found in {csv_path}")

    # Keep only required columns
    df = df[BASE_COLUMNS].copy()

    # Add standardized fields
    df["source_dataset"] = dataset_name
    df["class_label"] = dataset_cfg["class_label"]

    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

if VERBOSE:
    print("Combined dataset shape:", combined_df.shape)

    print("\nCounts by source_dataset:")
    print(combined_df["source_dataset"].value_counts())

    print("\nCounts by class_label:")
    print(combined_df["class_label"].value_counts())

    print("\nColumns in combined dataset:")
    print(combined_df.columns.tolist())

combined_csv_path = Path(COMBINED_METADATA_PATH)
combined_df.to_csv(combined_csv_path, index=False)

print(f"\nSaved combined metadata to: {combined_csv_path}")



### Step 5 — Train/Test Split, Save, and Validation

For each dataset independently:

1. Shuffle rows using a fixed random seed  
2. Split into exact counts:
   - **train = 2400**
   - **test = 600**

Then:

* Combine all training subsets
* Combine all testing subsets
* Shuffle each subset independently

Save:

* `train_metadata.csv`
* `test_metadata.csv`

Finally:

* Validate file existence
* Report dataset sizes and distributions

In [ ]:
# =========================
# Step 5: Split + Save + Validate Metadata
# =========================

split_dfs = []

# -------------------------------------------------
# Split each dataset into train/test
# -------------------------------------------------
for dataset_name, group_df in combined_df.groupby("source_dataset"):
    # Shuffle each dataset independently
    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # Exact-count split
    train_df = group_df.iloc[:TRAIN_PER_SOURCE].copy()
    test_df = group_df.iloc[
        TRAIN_PER_SOURCE:TRAIN_PER_SOURCE + TEST_PER_SOURCE
    ].copy()

    train_df["subset"] = TRAIN_SUBSET
    test_df["subset"] = TEST_SUBSET

    split_dfs.extend([train_df, test_df])

# -------------------------------------------------
# Combine all splits
# -------------------------------------------------
split_df = pd.concat(split_dfs, ignore_index=True)

train_metadata = split_df[split_df["subset"] == TRAIN_SUBSET].copy()
test_metadata = split_df[split_df["subset"] == TEST_SUBSET].copy()

# Final shuffle of each subset independently
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_metadata = test_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# -------------------------------------------------
# Save outputs
# -------------------------------------------------
train_csv_path = Path(TRAIN_METADATA_PATH)
test_csv_path = Path(TEST_METADATA_PATH)

train_metadata.to_csv(train_csv_path, index=False)
test_metadata.to_csv(test_csv_path, index=False)

# -------------------------------------------------
# Reporting
# -------------------------------------------------
if VERBOSE:
    print("Train shape:", train_metadata.shape)
    print("Test shape:", test_metadata.shape)

    print("\nCounts by subset:")
    print(pd.Series({
        TRAIN_SUBSET: len(train_metadata),
        TEST_SUBSET: len(test_metadata),
    }))

    print("\nCounts by subset and source_dataset:")
    print(split_df.groupby(["subset", "source_dataset"]).size())

    print("\nCounts by subset and class_label:")
    print(split_df.groupby(["subset", "class_label"]).size())

print(f"\nSaved train metadata to: {train_csv_path}")
print(f"Saved test metadata to: {test_csv_path}")

# -------------------------------------------------
# Validation of output CSV files
# -------------------------------------------------
print("\nValidation of saved CSV files:")

for csv_path in [
    Path(COMBINED_METADATA_PATH),
    Path(TRAIN_METADATA_PATH),
    Path(TEST_METADATA_PATH),
]:
    if not csv_path.exists():
        print(f"Missing: {csv_path}")
    else:
        df = pd.read_csv(csv_path)
        print(f"{csv_path.name}: {df.shape}")

